In [18]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# importerer data
model_ready_df = pd.read_csv("../output/model_ready.csv")

In [19]:
# sjekker at model_ready data stemmer
model_ready_df.head()

,timestamp,hour,day_of_week,month,station,free_bikes,free_bikes_next_hour,arrivals,departures,trips_count,temperature,precipitation,wind_speed
0,2024-04-10 23:00:00+02:00,23,2,4,Akvariet,0,0.0,3,3,14,5.1,0,17.4
1,2024-04-10 23:00:00+02:00,23,2,4,Dreggsallmenningen Sør,0,12.0,1,1,14,5.1,0,17.4
2,2024-04-10 23:00:00+02:00,23,2,4,Studentboligene,0,15.0,1,1,14,5.1,0,17.4
3,2024-04-10 23:00:00+02:00,23,2,4,Florida Bybanestopp,0,0.0,1,1,14,5.1,0,17.4
4,2024-04-10 23:00:00+02:00,23,2,4,Møllendalsplass,0,4.0,1,1,14,5.1,0,17.4


In [20]:
# deler data i mål- og prediktorvariabler og gjør om stations til å bruke one-hot encode
X = pd.get_dummies(model_ready_df.drop(columns=["free_bikes_next_hour", "timestamp"]),
                   columns=["station"], prefix="station")
y = model_ready_df["free_bikes_next_hour"]

In [21]:
# sjekker at X og y stemmer overens med data
X.head()

,hour,day_of_week,month,free_bikes,arrivals,departures,trips_count,temperature,precipitation,wind_speed,station_Akvariet,station_Damsgårdsveien 71,station_Dreggsallmenningen Sør,station_Florida Bybanestopp,station_Grieghallen,station_Høyteknologisenteret,station_Møllendalsplass,station_Studentboligene,station_Torgallmenningen
0,23,2,4,0,3,3,14,5.1,0,17.4,True,False,False,False,False,False,False,False,False
1,23,2,4,0,1,1,14,5.1,0,17.4,False,False,True,False,False,False,False,False,False
2,23,2,4,0,1,1,14,5.1,0,17.4,False,False,False,False,False,False,False,True,False
3,23,2,4,0,1,1,14,5.1,0,17.4,False,False,False,True,False,False,False,False,False
4,23,2,4,0,1,1,14,5.1,0,17.4,False,False,False,False,False,False,True,False,False


In [22]:
y.head()

0     0.0
1    12.0
2    15.0
3     0.0
4     4.0
Name: free_bikes_next_hour, dtype: float64

In [23]:
# deler opp i trenings-, validerings- og testdata
X_train, X_valtest, y_train, y_valtest = train_test_split(X, y, train_size=0.7, shuffle=False, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_valtest, y_valtest, train_size=0.5, shuffle=False, random_state=42)

In [24]:
# trener en grunnlinjemodell på treningsdata og finner RMSE på trenings- og valideringsdata
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_train_rmse = root_mean_squared_error(y_train, baseline.predict(X_train))
baseline_val_rmse = root_mean_squared_error(y_val, baseline.predict(X_val))

print("Resultat for baseline-modell:\n")
print("Treningsdata-RMSE-en er:", round(baseline_train_rmse, 3),
      "\nValideringsdata-RMSE-en er:", round(baseline_val_rmse, 3))

Resultat for baseline-modell:

Treningsdata-RMSE-en er: 7.1 
Valideringsdata-RMSE-en er: 5.059


In [25]:
# trener en lineær regresjonsmodell på treningsdata og finner RMSE på trenings- og valideringsdata
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
train_rmse_linreg_scores = root_mean_squared_error(y_train, lin_reg.predict(X_train))
val_rmse_linreg_scores = root_mean_squared_error(y_val, lin_reg.predict(X_val))

print("Resultat for lineær regresjon:\n")
print("Treningsdata-RMSE-en er:", round(train_rmse_linreg_scores, 3),
      "\nValideringsdata-RMSE-en er:", round(val_rmse_linreg_scores, 3))

Resultat for lineær regresjon:

Treningsdata-RMSE-en er: 1.38 
Valideringsdata-RMSE-en er: 1.094


In [26]:
# trener lasso-modell på ulike verdier av alpha mellom 0.01 og 10
lasso_models = {alpha: Lasso(alpha=alpha) for alpha in np.arange(0.01, 10, 0.1)}
train_rmse_scores = {}
val_rmse_scores = {}

for alpha, model in lasso_models.items():
    model.fit(X_train, y_train)
    train_rmse_scores[alpha] = root_mean_squared_error(y_train, model.predict(X_train))
    val_rmse_scores[alpha] = root_mean_squared_error(y_val, model.predict(X_val))

# finner hvilken alpha som gir laveste RMSE
best_alpha = min(val_rmse_scores, key=val_rmse_scores.get)

# finner modell for beste alpha
best_lasso_model = lasso_models[best_alpha]

# finner RMSE på trenings- og valideringsdata for beste alpha
best_alpha_train_rmse = train_rmse_scores[best_alpha]
best_alpha_val_rmse = val_rmse_scores[best_alpha]

print("Resultat for lasso-regresjon:\n")
print("Den beste alpha-en er", round(best_alpha, 3))
print("Den beste treningsdata-RMSE-en er:", round(best_alpha_train_rmse, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(best_alpha_val_rmse, 3))

Resultat for lasso-regresjon:

Den beste alpha-en er 0.41
Den beste treningsdata-RMSE-en er: 1.486 
Den beste valideringsdata-RMSE-en er: 1.033


In [27]:
# lager datasett med polynomielle data og utfører samme prosedyre med alpha
poly = PolynomialFeatures(degree=2)
X_train_pf = poly.fit_transform(X_train)
X_val_pf = poly.transform(X_val)

lasso_pf_models = {alpha: Lasso(alpha=alpha) for alpha in np.arange(0.01, 10, 0.1)}

train_rmse_scores_pf = {}
val_rmse_scores_pf = {}

for alpha, model in lasso_pf_models.items():
    model.fit(X_train_pf, y_train)
    train_rmse_scores_pf[alpha] = root_mean_squared_error(y_train, model.predict(X_train_pf))
    val_rmse_scores_pf[alpha] = root_mean_squared_error(y_val, model.predict(X_val_pf))

best_alpha_pf = min(val_rmse_scores_pf, key=val_rmse_scores_pf.get)
best_lasso_pf_model = lasso_pf_models[best_alpha_pf]

best_alpha_train_rmse_pf = train_rmse_scores_pf[best_alpha_pf]
best_alpha_val_rmse_pf = val_rmse_scores_pf[best_alpha_pf]

print("Resultat for lasso-regresjon med polynom av andre grad:\n")
print("Den beste alpha-en er", best_alpha_pf)
print("Den beste treningsdata-RMSE-en er:", round(best_alpha_train_rmse_pf, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(best_alpha_val_rmse_pf, 3))

Resultat for lasso-regresjon med polynom av andre grad:

Den beste alpha-en er 0.01
Den beste treningsdata-RMSE-en er: 1.283 
Den beste valideringsdata-RMSE-en er: 1.031


In [28]:
# visualiser også validerings-RMSE avhengig av alpha for begge lasso-modellene
alphas = list(lasso_models.keys())
alphas_pf = list(lasso_pf_models.keys())
val_rmse_lasso = [val_rmse_scores[a] for a in alphas]
val_rmse_lasso_pf = [val_rmse_scores_pf[a] for a in alphas_pf]

fig = px.line(x=alphas_pf*2, y=val_rmse_lasso+val_rmse_lasso_pf, color=["Val_lasso"] * len(alphas) + ["Val_lasso_pf"] * len(alphas_pf),
              markers=True, labels={"x":"alpha", "y":"RMSE", "color":"dataset"})
fig.show()

In [29]:
# prøver en k-nærmeste nabo modell for noen verdier av k
k_neighbors = {n_neighbors: KNeighborsRegressor(n_neighbors=n_neighbors) for n_neighbors in [1, 5, 10, 20, 30]}
train_rmse_scores_k_neighbors = {}
val_rmse_scores_k_neighbors = {}

for neighbor, model in k_neighbors.items():
    model.fit(X_train, y_train)
    train_rmse_scores_k_neighbors[neighbor] = root_mean_squared_error(y_train, model.predict(X_train))
    val_rmse_scores_k_neighbors[neighbor] = root_mean_squared_error(y_val, model.predict(X_val))

# finner beste antall naboer og modellen som bruker dette antallet
best_neighborcount = min(val_rmse_scores_k_neighbors, key=val_rmse_scores_k_neighbors.get)
best_k_neighbor_model = k_neighbors[best_neighborcount]

# finner RMSE på trenings- og valideringsdata for beste antall naboer
best_train_rmse_neighborcount = train_rmse_scores_k_neighbors[best_neighborcount]
best_val_rmse_neighborcount = val_rmse_scores_k_neighbors[best_neighborcount]

print("Resultat for k-nærmeste-nabo-regresjon:\n")
print("Det beste antall naboer er:", best_neighborcount)
print("Den beste treningsdata-RMSE-en er:", round(best_train_rmse_neighborcount, 3),
      "\nDen beste valideringsdata-RMSE-en er:", round(best_val_rmse_neighborcount, 3))
    

Resultat for k-nærmeste-nabo-regresjon:

Det beste antall naboer er: 5
Den beste treningsdata-RMSE-en er: 1.384 
Den beste valideringsdata-RMSE-en er: 1.305


In [30]:
# trener og finner RMSE på noen flere modeller som ikke krever spesielle hyperparametre
models = {"Elastic Net": ElasticNet(random_state=0),
          "Random Forest": RandomForestRegressor(random_state=0),
          "Support Vector": make_pipeline(StandardScaler(), SVR(kernel="linear")),
          "Ridge": Ridge()}

# finner RMSE på trenings- og valideringsdata for å finne beste av de enkle modellene
curr_train_rmse = 0
curr_val_rmse = 0

best_train_rmse = 10
best_val_rmse = 10

best_train_rmse_model = ""
best_overall_in_models = ""

for name, model in models.items():
    model.fit(X_train, y_train)
    curr_train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
    curr_val_rmse = root_mean_squared_error(y_val, model.predict(X_val))

    if (curr_train_rmse < best_train_rmse):
        best_train_rmse = curr_train_rmse
        best_train_rmse_model = name

    if (curr_val_rmse < best_val_rmse):
        best_val_rmse = curr_val_rmse
        best_overall_in_models = name


print("Den beste modellen på treningsdata er:", best_train_rmse_model)
print("Den beste modellen på valideringsdata, og dermed også totalt beste av disse modellene, er:", best_overall_in_models)

Den beste modellen på treningsdata er: Random Forest
Den beste modellen på valideringsdata, og dermed også totalt beste av disse modellene, er: Support Vector


In [31]:
train_rmse_models_scores = root_mean_squared_error(y_train, models[best_overall_in_models].predict(X_train))
val_rmse_models_scores = root_mean_squared_error(y_val, models[best_overall_in_models].predict(X_val))

print("Resultat for beste av en rekke modeller:", best_overall_in_models,"\n")
print("Treningsdata-RMSE-en er:", round(train_rmse_models_scores, 3),
      "\nValideringsdata-RMSE-en er:", round(val_rmse_models_scores, 3))

Resultat for beste av en rekke modeller: Support Vector 

Treningsdata-RMSE-en er: 1.509 
Valideringsdata-RMSE-en er: 1.036


In [32]:
# finner modellen med best test-RMSE ut ifra tabellen
results = pd.DataFrame([
    {"model": baseline, "train_rmse": baseline_train_rmse, "val_rmse": baseline_val_rmse},
    {"model": lin_reg, "train_rmse": train_rmse_linreg_scores, "val_rmse": val_rmse_linreg_scores},
    {"model": best_lasso_model, "train_rmse": best_alpha_train_rmse, "val_rmse": best_alpha_val_rmse},
    {"model": best_lasso_pf_model, "train_rmse": best_alpha_train_rmse_pf, "val_rmse": best_alpha_val_rmse_pf},
    {"model": best_k_neighbor_model, "train_rmse": best_train_rmse_neighborcount, "val_rmse": best_val_rmse_neighborcount},
    {"model": best_overall_in_models, "train_rmse": train_rmse_models_scores, "val_rmse": val_rmse_models_scores}])

print(round(results, 3))

best_val_model = results.loc[results["val_rmse"].idxmin()]["model"]

if (best_val_model == best_lasso_pf_model):
    X_test_pf = poly.transform(X_test)
    print("\nModellen med best RMSE på valideringsdata er:\n Poly + Lasso, med generaliseringsevne:",
          round(root_mean_squared_error(y_test, best_val_model.predict(X_test_pf)), 3))
else:
    print("\nModellen med best RMSE på valideringsdata er:\n", best_val_model,
          "med generaliseringsevne:",
          round(root_mean_squared_error(y_test, best_val_model.predict(X_test)), 3))

                                          model  train_rmse  val_rmse
0                              DummyRegressor()       7.100     5.059
1                            LinearRegression()       1.380     1.094
2  Lasso(alpha=np.float64(0.41000000000000003))       1.486     1.033
3                 Lasso(alpha=np.float64(0.01))       1.283     1.031
4                         KNeighborsRegressor()       1.384     1.305
5                                Support Vector       1.509     1.036

Modellen med best RMSE på valideringsdata er:
 Poly + Lasso, med generaliseringsevne: 1.784
